# EVA Cesar · 01. Diagnóstico inicial del CSV de EVA

Programación en Lenguajes Estadísticos · UNAL

## Montaje de Drive y ubicación del proyecto

Ejecutar desde la raíz del proyecto o indicar `EVA_RAIZ`. Correr los cuadernos en orden 01, 02 y 03.

In [ ]:
from pathlib import Path
import os

ruta_local = Path(os.environ.get("EVA_RAIZ", ".")).resolve()
try:
    from google.colab import drive
except ImportError:
    raiz = ruta_local
else:
    drive.mount("/content/drive")
    rutas = [ruta_local, Path("/content/drive/MyDrive/proyecto_eva_cesar"),
             Path("/content/drive/Shareddrives/Programacion en lenguajes estadisticos/proyecto_eva_cesar")]
    raiz = next((p for p in rutas if (p / "datos/datos_crudos").is_dir()), rutas[1])
if not (raiz / "datos/datos_crudos").is_dir():
    raise FileNotFoundError(f"Indique la carpeta raíz de EVA con EVA_RAIZ. Ruta revisada: {raiz}")
docs = raiz / "documentos"
limpios = raiz / "datos/datos_limpios"
figuras = docs / "figuras"
for carpeta in (docs, limpios, figuras):
    carpeta.mkdir(parents=True, exist_ok=True)


## Inspección del CSV nacional

In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

archivo = raiz / "datos/datos_crudos/Evaluaciones_Agropecuarias_Municipales_EVA_20261028.csv"
crudo = pd.read_csv(archivo, dtype="string", keep_default_na=False, na_values=["", "NA", "N/A", "NULL"])
def normalizar(nombre):
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sin_tildes.lower()).strip("_")

crudo.columns = crudo.columns.map(normalizar)
crudo["ano"] = pd.to_numeric(crudo["ano"].str.replace(".", "", regex=False)).astype("Int64")
eva = crudo.loc[crudo["departamento"].str.strip().str.upper().eq("CESAR") & crudo["ano"].between(2006, 2018)].copy()
assert len(eva) > 0
print(f"Base nacional: {len(crudo):,} filas. Cesar: {len(eva):,} filas.")
del crudo
medidas = ["area_sembrada_ha", "area_cosechada_ha", "produccion_t", "rendimiento_t_ha"]
for columna in medidas:
    eva[columna] = pd.to_numeric(eva[columna].str.replace(r"[\s.]", "", regex=True)
                               .str.replace(",", ".", regex=False)).astype(float)
for columna in ["cod_dep", "cod_mun"]:
    eva[columna] = eva[columna].str.replace(".", "", regex=False)

display(eva.head(), eva[medidas].describe())
eva.info()

## Faltantes, duplicados y cobertura

In [ ]:
def exportar(tabla, carpeta, nombre):
    tabla.to_csv(carpeta / nombre, index=False)

clave = ["cod_mun", "cultivo", "ano", "periodo", "desagregacion_regional_y_o_sistema_productivo"]
tipos = pd.DataFrame({"variable": eva.columns, "tipo": eva.dtypes.astype(str).values,
                      "distintos": eva.nunique().values})
faltantes = pd.DataFrame({"variable": eva.columns, "n_na": eva.isna().sum().values})
faltantes["pct_na"] = 100 * faltantes["n_na"] / len(eva)
duplicados = pd.DataFrame({"criterio": ["filas_identicas", "claves_repetidas"],
                          "n": [eva.duplicated().sum(), eva.duplicated(clave).sum()]})
cobertura = eva.groupby("ano", as_index=False).agg(
    registros=("cultivo", "size"), rendimiento_faltante=("rendimiento_t_ha", lambda x: x.isna().sum()),
    municipios=("cod_mun", "nunique"), cultivos=("cultivo", "nunique"))
periodo = eva["periodo"].str.replace(r"^\d{4}$", "ANUAL", regex=True).str.replace(r"^\d{4}([AB])$", r"\1", regex=True)
periodos = pd.crosstab(eva["ano"], periodo).stack().rename("registros").rename_axis(["ano", "periodo"]).reset_index()

for nombre, tabla in {"tipos": tipos, "faltantes": faltantes, "duplicados": duplicados,
                      "cobertura": cobertura, "periodos": periodos}.items():
    exportar(tabla, docs, f"diagnostico_Python_{nombre}.csv")
exportar(eva, limpios, "eva_cesar_2006_2018_sin_limpiar_py.csv")
display(cobertura, periodos, duplicados)

## Primera visualización antes de limpiar

La transformación log10(1 + x) conserva los ceros y permite comparar el rango de cada magnitud. El rendimiento mezcla productos físicamente distintos, por lo que en la etapa siguiente se describe dentro de cada cultivo.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, ejes = plt.subplots(2, 2, figsize=(9, 6), layout="constrained")
titulos = ["Área sembrada (ha)", "Área cosechada (ha)", "Producción (t)", "Rendimiento (t/ha)"]
for variable, titulo, eje in zip(medidas, titulos, ejes.flat):
    valores = eva.loc[eva[variable].ge(0), variable]
    eje.hist(np.log10(1 + valores), bins=30, color="#6D45A6", edgecolor="white")
    eje.set(title=titulo, xlabel="log10(1 + valor)", ylabel="Registros")
fig.savefig(figuras / "eva_Py_01_inspeccion.png", dpi=180)
plt.show()